# Actualización de Tablas Vida

## 1. Importación de paqueterías

In [1]:
import pandas as pd
import urllib
from sqlalchemy import create_engine, text


## 2. Conexión a SQL Server

In [2]:

# -----------------------------
# 1️⃣ Conexión a SQL Server Azure
# -----------------------------
params_azure = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=ikaldb.database.windows.net;"
    "DATABASE=actuarial;"
    "UID=CloudSA0052c2f7;"
    "PWD=ricostamales01#;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)
engine_azure = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_azure}",
    fast_executemany=True
)
print("✅ Conexión SQL Server AZURE establecida.")

# -----------------------------
# 2️⃣ Conexión a SQL Server Local
# -----------------------------
params_local = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=IKAL14\\SQLEXPRESS;"
    "DATABASE=actuarial;"
    "Trusted_Connection=yes;"
)
engine_local = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_local}",
    fast_executemany=True
)
print("✅ Conexión SQL Server Local establecida.")


✅ Conexión SQL Server AZURE establecida.
✅ Conexión SQL Server Local establecida.


## 3. Carga de datos

In [3]:
# -----------------------------
# 2️⃣ Leer Excel con datos nuevos
# -----------------------------
archivo_excel = r"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/2025/202509/202509_Siniestros_Marine_PROCESADO.xlsx"
df_nuevo = pd.read_excel(archivo_excel)
print(f"Datos del Excel '{archivo_excel}' leídos correctamente.")

Datos del Excel 'C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/2025/202509/202509_Siniestros_Marine_PROCESADO.xlsx' leídos correctamente.


## 4. Limpieza y carga de tabla transporte

In [4]:

# -----------------------------
# 3️⃣ Actualizar tabla 'transporte'
# -----------------------------
print("📊 Actualizando tabla 'transporte' (mes actual)...")
df_nuevo.to_sql('transporte', con=engine_azure, if_exists='replace', index=False, schema='dbo')
df_nuevo.to_sql('transporte', con=engine_local, if_exists='replace', index=False, schema='dbo')
print(f"✅ Tabla 'transporte' reemplazada con {len(df_nuevo):,} registros")


📊 Actualizando tabla 'transporte' (mes actual)...
✅ Tabla 'transporte' reemplazada con 4,251 registros


## 5. Inserción y actualización de tabla transportehist

In [10]:

# =============================================================================
# 5️⃣ INSERCIÓN Y ACTUALIZACIÓN DE TABLA TRANSPORTEHIST CON MERGE
# =============================================================================

# 📋 PASO 1: Obtener estructura de tabla SQL existente
print("\n" + "="*80)
print("PASO 1: Obtener estructura de tabla SQL")
print("="*80)

query_columnas = """
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = 'transportehist' AND TABLE_SCHEMA = 'dbo'
ORDER BY ORDINAL_POSITION
"""
df_cols_azure = pd.read_sql(query_columnas, con=engine_azure)
df_cols_local = pd.read_sql(query_columnas, con=engine_local)

columnas_sql_list = df_cols_azure['COLUMN_NAME'].tolist()
columnas_sql_upper = {c.upper(): c for c in columnas_sql_list}

# Columnas de fecha
columnas_fecha_sql = set(df_cols_azure[df_cols_azure['DATA_TYPE'].isin(['date', 'datetime', 'datetime2'])]['COLUMN_NAME'].tolist())

# max_len_map: mínimo entre Azure y Local (límite más restrictivo)
def _get_len_map(df):
    m = (
        df['DATA_TYPE'].isin(['varchar', 'nvarchar', 'char', 'nchar']) &
        df['CHARACTER_MAXIMUM_LENGTH'].notna() &
        (df['CHARACTER_MAXIMUM_LENGTH'] > 0)
    )
    return dict(zip(df.loc[m, 'COLUMN_NAME'], df.loc[m, 'CHARACTER_MAXIMUM_LENGTH'].astype(int)))

len_azure = _get_len_map(df_cols_azure)
len_local = _get_len_map(df_cols_local)
all_text_cols = set(len_azure) | set(len_local)
max_len_map = {
    col: min(v for v in [len_azure.get(col), len_local.get(col)] if v is not None)
    for col in all_text_cols
}

# Leer PK real de la tabla desde la base de datos
query_pk = """
SELECT c.COLUMN_NAME
FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS tc
JOIN INFORMATION_SCHEMA.CONSTRAINT_COLUMN_USAGE c
    ON tc.CONSTRAINT_NAME = c.CONSTRAINT_NAME AND tc.TABLE_SCHEMA = c.TABLE_SCHEMA
WHERE tc.TABLE_NAME = 'transportehist'
    AND tc.TABLE_SCHEMA = 'dbo'
    AND tc.CONSTRAINT_TYPE = 'PRIMARY KEY'
ORDER BY c.COLUMN_NAME
"""
columnas_clave = ['CLAIMS-ID','GROSS RESERVE']

print(f"✓ Columnas en tabla SQL (transportehist): {len(columnas_sql_list)}")
print(f"✓ Columnas de tipo fecha: {sorted(columnas_fecha_sql)}")
print(f"✓ Columnas de texto con límite de longitud: {len(max_len_map)}")
print(f"✓ Columnas PK detectadas: {columnas_clave}")

# Renombrar columnas del Excel al nombre exacto de SQL (case-insensitive)
rename_map = {
    col: columnas_sql_upper[col.upper()]
    for col in df_nuevo.columns
    if col.upper() in columnas_sql_upper and col != columnas_sql_upper[col.upper()]
}
df_nuevo_renamed = df_nuevo.rename(columns=rename_map)
print(f"✓ Columnas renombradas para coincidir con SQL: {len(rename_map)}")

columnas_sql = set(columnas_sql_list)
columnas_excel = set(df_nuevo_renamed.columns.tolist())
columnas_validas = sorted(list(columnas_excel & columnas_sql))
print(f"✓ Columnas válidas para MERGE: {len(columnas_validas)}")

solo_excel = columnas_excel - columnas_sql
if solo_excel:
    print(f"⚠️  Columnas en Excel pero NO en SQL ({len(solo_excel)}): {sorted(list(solo_excel))}")

# 📋 PASO 2: Preparar datos del Excel
print("\n" + "="*80)
print("PASO 2: Preparar datos del Excel")
print("="*80)

# Validar que las columnas PK existen en los datos
faltantes_clave = [c for c in columnas_clave if c not in columnas_validas]
if faltantes_clave:
    raise ValueError(f"❌ Columnas PK faltantes en los datos: {faltantes_clave}")
print(f"✓ Columnas PK presentes: {columnas_clave}")

df_transporte_limpio = df_nuevo_renamed[columnas_validas].copy()

# Convertir columnas de fecha (maneja datetime y enteros seriales de Excel)
cols_fecha_presentes = [c for c in columnas_fecha_sql if c in df_transporte_limpio.columns]
for col in cols_fecha_presentes:
    if pd.api.types.is_integer_dtype(df_transporte_limpio[col]) or pd.api.types.is_float_dtype(df_transporte_limpio[col]):
        df_transporte_limpio[col] = pd.to_datetime(df_transporte_limpio[col], unit='D', origin='1899-12-30', errors='coerce')
    else:
        df_transporte_limpio[col] = pd.to_datetime(df_transporte_limpio[col], errors='coerce')
print(f"✓ Columnas de fecha convertidas: {cols_fecha_presentes}")

# Truncar columnas de texto que excedan el límite de SQL
cols_truncadas = []
for col, max_len in max_len_map.items():
    if col in df_transporte_limpio.columns and df_transporte_limpio[col].dtype == object:
        mask = df_transporte_limpio[col].str.len() > max_len
        if mask.any():
            df_transporte_limpio[col] = df_transporte_limpio[col].str[:max_len]
            cols_truncadas.append(f"{col}({max_len})")
if cols_truncadas:
    print(f"✓ Columnas truncadas al límite SQL: {cols_truncadas}")

# 📋 PASO 3: Cargar datos a tabla temporal
print("\n" + "="*80)
print("PASO 3: Cargar datos a tabla temporal")
print("="*80)

df_transporte_limpio.to_sql('temp_transportehist_mes', con=engine_local, if_exists='replace', index=False, schema='dbo')
df_transporte_limpio.to_sql('temp_transportehist_mes', con=engine_azure, if_exists='replace', index=False, schema='dbo')
print(f"✓ Tabla temporal 'temp_transportehist_mes' creada con {len(df_transporte_limpio):,} registros")

# 📋 PASO 4: Construir y ejecutar MERGE
print("\n" + "="*80)
print("PASO 4: Ejecutar MERGE (actualización/inserción)")
print("="*80)

on_clause = ' AND '.join([f'target.[{col}] = source.[{col}]' for col in columnas_clave])
insert_cols = ', '.join([f'[{col}]' for col in columnas_validas])
insert_vals = ', '.join([f'source.[{col}]' for col in columnas_validas])
update_cols = [col for col in columnas_validas if col not in columnas_clave]
update_set = ', '.join([f'target.[{col}] = source.[{col}]' for col in update_cols])

merge_query = f"""
MERGE transportehist AS target
USING temp_transportehist_mes AS source
ON ({on_clause})

WHEN MATCHED THEN
    UPDATE SET {update_set}

WHEN NOT MATCHED THEN
    INSERT ({insert_cols})
    VALUES ({insert_vals})

OUTPUT $action AS Action;
"""

with engine_local.begin() as conn:
    result = conn.execute(text(merge_query))
    merge_output_local = result.fetchall()

with engine_azure.begin() as conn:
    result = conn.execute(text(merge_query))
    merge_output_azure = result.fetchall()

updates = sum(1 for row in merge_output_azure if row[0] == 'UPDATE')
inserts = sum(1 for row in merge_output_azure if row[0] == 'INSERT')

print(f"✅ MERGE completado:")
print(f"   • Registros actualizados: {updates:,}")
print(f"   • Registros nuevos (insertados): {inserts:,}")

# 📋 PASO 5: Resumen final
print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)

registros_despues_query = "SELECT COUNT(*) as total FROM transportehist"
registros_despues_azure = pd.read_sql(registros_despues_query, con=engine_azure)['total'][0]
registros_despues_local = pd.read_sql(registros_despues_query, con=engine_local)['total'][0]

print(f"\n✅ ESTADÍSTICAS:")
print(f" • Registros en transportehist ANTES: {registros_antes:,}")
print(f" • Registros procesados del Excel: {len(df_transporte_limpio):,}")
print(f" • Registros ACTUALIZADOS: {updates:,}")
print(f" • Registros NUEVOS insertados: {inserts:,}")
print(f" • Registros en transportehist DESPUÉS en Azure: {registros_despues_azure:,}")
print(f" • Registros en transportehist DESPUÉS en Local: {registros_despues_local:,}")

# Limpiar tabla temporal
with engine_local.begin() as conn:
    conn.execute(text("DROP TABLE temp_transportehist_mes"))
print(f"\n🧹 Tabla temporal 'temp_transportehist_mes' eliminada en Local")

with engine_azure.begin() as conn:
    conn.execute(text("DROP TABLE temp_transportehist_mes"))
print(f"🧹 Tabla temporal 'temp_transportehist_mes' eliminada en Azure")
print("="*80)



PASO 1: Obtener estructura de tabla SQL
✓ Columnas en tabla SQL (transportehist): 38
✓ Columnas de tipo fecha: ['ACCIDENT YEAR', 'DATE OF LOSS', 'DATE OF REPORT', 'MES CARGA', 'POLICY PERIOD END DATE', 'POLICY PERIOD START DATE', 'UW YEAR', 'YEAR OF THE RESERVE']
✓ Columnas de texto con límite de longitud: 13
✓ Columnas PK detectadas: ['CLAIMS-ID', 'GROSS RESERVE']
✓ Columnas renombradas para coincidir con SQL: 14
✓ Columnas válidas para MERGE: 38

PASO 2: Preparar datos del Excel
✓ Columnas PK presentes: ['CLAIMS-ID', 'GROSS RESERVE']
✓ Columnas de fecha convertidas: ['ACCIDENT YEAR', 'UW YEAR', 'POLICY PERIOD END DATE', 'MES CARGA', 'DATE OF LOSS', 'DATE OF REPORT', 'POLICY PERIOD START DATE', 'YEAR OF THE RESERVE']

PASO 3: Cargar datos a tabla temporal
✓ Tabla temporal 'temp_transportehist_mes' creada con 4,251 registros

PASO 4: Ejecutar MERGE (actualización/inserción)
✅ MERGE completado:
   • Registros actualizados: 0
   • Registros nuevos (insertados): 4,251

RESUMEN FINAL

✅ E